# Классификация: подробное изложение

Как превратить признаки объекта в решение о его классе? Последовательно построим линейную модель, получим логистическую потерю из вероятностных предположений, сравним её с SVM и разберём, что именно измеряют метрики.

Содержание:
1. Постановка задачи: оценка, вероятность и решение.
2. Логистическая регрессия: модель, правдоподобие, оптимизация.
3. SVM: геометрия зазора, ограничения и функции потерь.
4. Нелинейные границы: новые признаки и ядра.
5. Метрики: решения, ранжирование, вероятности.
6. Проверка качества на новых данных.

Основной маршрут рассчитан на последовательное объяснение идей. Раздел 7 содержит дополнительные подробности о выпуклости и нормировке регуляризации; их можно использовать при обсуждении вопросов. Развёрнутые примеры позволяют выбирать глубину объяснения, а не требуют разобрать весь текст за 20–30 минут.

[Краткий конспект](2.%20Classification%20%E2%80%94%20theory%20notes.ipynb) · [Практикум](2.%20Classification%2C%20Logistic%20Regression%2C%20Validation.ipynb)

## 1. Что предсказывает классификатор
### 1.1. Объекты, метки и цель обучения
Пусть дан набор наблюдений
$$\mathcal D=\{(x_i,y_i)\}_{i=1}^{n},\qquad x_i\in\mathbb R^d,\quad y_i\in\{0,1\}.$$
Вектор $x_i$ содержит признаки объекта, а $y_i$ — его известный класс. Например, для письма признаки описывают текст, а метка указывает, является ли оно спамом.

Класс 1 будем называть **положительным**: это обозначение интересующего события, а не оценка «хорошо или плохо». Для поиска спама положительный класс может означать именно спам.

Нужно найти правило, которое работает на **новых** объектах. В стандартной постановке обучающие и новые наблюдения независимо приходят из одного распределения. Малое число ошибок на обучении само по себе этой цели не обеспечивает.

В отличие от регрессии, итоговый ответ здесь дискретен. Однако внутри алгоритма удобно вычислять непрерывное число: оно позволяет выражать степень предпочтения класса и оптимизировать гладкую функцию.

### 1.2. Три разных результата: score, вероятность, класс
Различаем три величины:

| Величина | Пример | Смысл |
|---|---|---|
| Оценка, или score, $f(x)$ | $2.3$ | Насколько модель предпочитает класс 1 |
| Вероятность $p(x)$ | $0.91$ | Оценка $P(y=1\mid x)$ |
| Предсказанная метка $\hat y(x)$ | $1$ | Принятое решение |

Больший score должен соответствовать большей склонности к классу 1. Но значение $f(x)=2.3$ **не является вероятностью**. Не у каждого классификатора вообще есть вероятностная интерпретация выхода.

Если модель оценивает вероятность, метку получаем по порогу:
$$\hat y_t(x)=\mathbf1\{p(x)\ge t\}.$$
При $p(x)=0.4$ порог $t=0.5$ даёт класс 0, а порог $t=0.3$ — класс 1. Оценка модели не изменилась; изменилось правило принятия решения.

Это различие понадобится при выборе метрик: accuracy оценивает метки, ROC AUC — порядок scores, log loss — вероятности.

### 1.3. Линейная оценка и граница решения
Линейная модель вычисляет
$$z(x)=w^\top x+b=\sum_{j=1}^{d}w_jx_j+b.$$
Коэффициенты $w$ определяют вклад признаков, $b$ — сдвиг. При правиле $\hat y=\mathbf1\{z\ge0\}$ граница классов задаётся уравнением
$$w^\top x+b=0.$$
В двух измерениях это прямая, в трёх — плоскость, в общем случае — гиперплоскость. Вектор $w$ перпендикулярен границе; $b$ позволяет сдвинуть её от начала координат.

Например, при $z=2x_1-x_2-1$ граница имеет вид $x_2=2x_1-1$. Точка $(1,0)$ получает положительный score $1$, а точка $(0,0)$ — отрицательный score $-1$.

Здесь уже зафиксировано ограничение модели: одной гиперплоскости может быть недостаточно. Более точная оптимизация не расширяет множество допустимых границ.

## 2. Логистическая регрессия
### 2.1. Почему линейную оценку нельзя сразу назвать вероятностью
Число $z=w^\top x+b$ может быть любым действительным, тогда как вероятность должна лежать между 0 и 1. Используем **сигмоиду**:
$$p(x)=\sigma(z)=\frac{1}{1+e^{-z}}.$$
Она монотонно возрастает: $\sigma(0)=1/2$, $\sigma(2)\approx0.88$, $\sigma(-2)\approx0.12$. Большой положительный score даёт вероятность, близкую к 1, большой отрицательный — близкую к 0.

Более содержательная запись того же предположения:
$$\frac{p}{1-p}=e^z,\qquad \log\frac{p}{1-p}=z=w^\top x+b.$$
Отношение $p/(1-p)$ называется **шансами** события. Модель предполагает, что логарифм шансов линейно зависит от признаков. При увеличении $x_j$ на единицу и фиксированных остальных признаках шансы умножаются на $e^{w_j}$.

При пороге $1/2$ граница остаётся $z=0$. Сигмоида придаёт выходу вероятностный смысл, но сама по себе не превращает прямую границу в кривую.

### 2.2. От вероятностной модели к функции потерь
Условно на признаках метка имеет распределение Бернулли:
$$P(y_i\mid x_i;w,b)=p_i^{y_i}(1-p_i)^{1-y_i},\qquad p_i=\sigma(w^\top x_i+b).$$
Если $y_i=1$, вероятность наблюдённой метки равна $p_i$; если $y_i=0$, она равна $1-p_i$.

При условной независимости наблюдений правдоподобие — произведение этих вероятностей:
$$L(w,b)=\prod_{i=1}^{n}p_i^{y_i}(1-p_i)^{1-y_i}.$$
**Метод максимального правдоподобия** выбирает параметры, при которых фактически полученные метки наиболее правдоподобны. Логарифм сохраняет положение максимума и заменяет произведение суммой. Меняя знак и деля на $n$, получаем задачу минимизации:
$$J(w,b)=-\frac1n\log L
=\frac1n\sum_i\underbrace{\left[-y_i\log p_i-(1-y_i)\log(1-p_i)\right]}_{\ell(y_i,p_i)}.$$
Это **бинарная кросс-энтропия**, или **log loss**. Деление на $n$ не меняет минимум нерегуляризованной задачи, но делает масштаб критерия независимым от числа объектов.

Для положительного объекта $y=1$ потеря равна $-\log p$:

| Предсказанная вероятность $p$ | Потеря |
|---:|---:|
| $0.9$ | $0.105$ |
| $0.5$ | $0.693$ |
| $0.1$ | $2.303$ |

Уверенная ошибка обходится значительно дороже неуверенного ответа. Для отрицательного объекта ситуация симметрична: вместо $p$ используется $1-p$.

### 2.3. Почему оптимизируем log loss, а не число ошибок
Ошибка классификации одного объекта — индикатор $\mathbf1\{\hat y\ne y\}$. При небольшом изменении весов метка обычно не меняется, поэтому такой критерий почти всюду постоянен по параметрам и не даёт полезного градиента.

Log loss зависит от непрерывного score и меняется даже тогда, когда метка остаётся прежней. Для положительного объекта вероятности 0.6 и 0.99 дают одинаковый правильный класс, но разные потери. Поэтому градиентный метод получает сигнал, в каком направлении менять параметры.

Перепишем потерю через $z$. Так как
$$\log p=z-\log(1+e^z),\qquad \log(1-p)=-\log(1+e^z),$$
получаем
$$\ell(y,z)=\log(1+e^z)-yz.$$
Эта запись удобна для дифференцирования. Для вычислений нельзя буквально брать `exp(z)` при очень большом $z$: возникает переполнение. С меткой $s=2y-1$ эквивалентная устойчивая форма имеет вид
$$\ell(y,z)=\log(1+e^{-sz}),$$
и вычисляется как `np.logaddexp(0, -s*z)`. Её реализация — первая задача практикума.

### 2.4. Градиент и регуляризация
Производная по score особенно проста:
$$\frac{\partial\ell}{\partial z}=\frac{e^z}{1+e^z}-y=p-y.$$
По правилу цепочки $\partial z/\partial w=x$ и $\partial z/\partial b=1$. Поэтому
$$\nabla_w J=\frac1n\sum_i(p_i-y_i)x_i,\qquad
\frac{\partial J}{\partial b}=\frac1n\sum_i(p_i-y_i).$$
Для $y_i=1$ величина $p_i-y_i$ отрицательна: локально уменьшение потери требует увеличить score. Для $y_i=0$ — уменьшить его. Обновление параметров учитывает вклады всех объектов одновременно.

Добавим L2-регуляризацию:
$$J_\lambda(w,b)=J(w,b)+\frac\lambda2\|w\|^2.$$
Она штрафует большие коэффициенты. В принятых здесь соглашениях сдвиг $b$ не штрафуется:
$$w\leftarrow w-\eta\left[\frac1nX^\top(p-y)+\lambda w\right],\qquad
b\leftarrow b-\eta\frac1n\sum_i(p_i-y_i).$$
Здесь $X\in\mathbb R^{n\times d}$, $p,y\in\mathbb R^n$, а $\eta>0$ — шаг градиентного спуска. Параметр $\lambda$ определяет задачу оптимизации; $\eta$ определяет способ её решения. Они выполняют разные роли.

Смысл штрафа зависит от масштаба признаков: при умножении признака на 100 тот же вклад в score можно получить коэффициентом в 100 раз меньше. Поэтому перед регуляризованными линейными моделями обычно стандартизуют признаки, обучая преобразование только по train.

### 2.5. Почему accuracy = 1 не останавливает обучение
Пусть данные строго разделимы: существует $(v,a)$, для которого
$$s_i(v^\top x_i+a)>0\quad\text{для всех }i.$$
Умножим оба параметра на $c>0$. Знаки scores и предсказанные классы не изменятся. Но потери равны
$$\ell_i(c)=\log\left(1+\exp[-c\,s_i(v^\top x_i+a)]\right)\longrightarrow0
\quad\text{при }c\to\infty.$$
При любом конечном $c$ каждая потеря положительна. Значит, на строго разделимой выборке без регуляризации инфимум равен нулю, но конечного минимума нет: веса могут продолжать расти после достижения идеальной accuracy.

L2 добавляет цену за рост $\|w\|$. Если же одинаковые признаки встречаются с противоположными метками, нельзя одновременно сделать обе потери малыми: увеличение уверенности в одной метке ухудшает другую.

Это объясняет, почему во втором упражнении отдельно наблюдаем accuracy, log loss и норму весов: они описывают разные стороны обучения.

## 3. Метод опорных векторов: SVM
### 3.1. Как выбрать одну из многих разделяющих прямых
Если данные разделимы, правильные метки могут давать разные гиперплоскости. SVM выбирает ту, которая оставляет наибольший **зазор** до ближайших объектов классов.

Для этого используем метки $s_i\in\{-1,+1\}$ и score $f(x)=w^\top x+b$. **Функциональный отступ**:
$$m_i=s_i f(x_i).$$
Если $m_i>0$, объект классифицирован правильно; если $m_i<0$, неправильно. Но величину отступа можно искусственно увеличить, умножив $w,b$ на одно положительное число, без изменения границы.

Чтобы получить геометрическую величину, делим на длину нормали:
$$r_i=\frac{s_i(w^\top x_i+b)}{\|w\|},\qquad w\ne0.$$
Это подписанное расстояние до гиперплоскости, положительное с правильной стороны. При совместном положительном масштабировании $w,b$ оно не меняется.

### 3.2. Hard-margin SVM
На строго разделимых данных масштабируем параметры так, чтобы минимальный функциональный отступ был равен 1. Тогда ограничения имеют вид
$$s_i(w^\top x_i+b)\ge1\quad\text{для всех }i.$$
Ближайшие объекты лежат на опорных гиперплоскостях $f(x)=+1$ и $f(x)=-1$. Расстояние каждой из них до границы $f(x)=0$ равно $1/\|w\|$, а ширина полосы между ними — $2/\|w\|$.

Максимизация ширины полосы эквивалентна минимизации нормы:
$$\min_{w,b}\frac12\|w\|^2
\quad\text{при}\quad s_i(w^\top x_i+b)\ge1.$$
Квадрат нормы и множитель $1/2$ удобны для оптимизации и не меняют решение. Объекты, лежащие на границе зазора, определяют разделитель; в двойственной формулировке опорными называют объекты с ненулевыми множителями при ограничениях.

Если классы пересекаются, эти ограничения несовместны. Даже один ошибочно размеченный объект может сделать hard-margin постановку неприменимой.

### 3.3. Soft-margin SVM и hinge loss
Разрешим каждому объекту нарушать ограничение на величину $\xi_i\ge0$:
$$s_i(w^\top x_i+b)\ge1-\xi_i.$$
Теперь задача имеет вид
$$\min_{w,b,\xi}\frac12\|w\|^2+C\sum_i\xi_i,
\qquad \xi_i\ge0.$$
Норма отвечает за широкий зазор, сумма $\xi_i$ — за нарушения. Большое $C$ делает нарушения более дорогими.

Для фиксированных $w,b$ минимальное допустимое значение
$$\xi_i=\max(0,1-m_i).$$
Подставляя его, получаем **hinge loss**:
$$\min_{w,b}\frac12\|w\|^2+C\sum_i\underbrace{\max(0,1-m_i)}_{\ell_{hinge}(m_i)}.$$

| Отступ | Положение объекта | Hinge loss |
|---|---|---|
| $m_i\ge1$ | Правильно, вне внутренней части полосы | $0$ |
| $0<m_i<1$ | Правильно, но внутри полосы | $1-m_i$ |
| $m_i<0$ | Неправильно | $1-m_i>1$ |

Следовательно, нарушение зазора не обязательно означает ошибку класса. Например, $m=0.4$ даёт правильную метку, но положительную потерю 0.6.

### 3.4. Сравнение с логистической регрессией
Обе модели могут использовать одинаковый линейный score $f(x)=w^\top x+b$. Различается прежде всего критерий обучения:
$$\ell_{log}(m)=\log(1+e^{-m}),\qquad
\ell_{hinge}(m)=\max(0,1-m).$$
При $m=2$ объект уверенно классифицирован правильно. Hinge loss уже равна нулю, а логистическая потеря примерно 0.127 и продолжает уменьшаться при росте отступа. При $m=-2$ обе потери велики: объект находится с неправильной стороны.

Log loss гладкая; hinge loss имеет излом в $m=1$. Логистическая регрессия задаёт модель условной вероятности. SVM в стандартном виде выдаёт оценку для разделения и ранжирования; `decision_function` нельзя интерпретировать как вероятность.

Эти различия не означают, что один метод всегда лучше другого. Качество зависит от данных, признаков, регуляризации и выбранной метрики.

## 4. Как получить нелинейную границу
### 4.1. Явное преобразование признаков: пример XOR
Пусть $y=1$, если $x_1,x_2$ имеют разные знаки, и $y=0$, если одинаковые. Положительный класс занимает два противоположных квадранта. Одна прямая не разделяет их с двумя оставшимися квадрантами.

Добавим признак
$$u=x_1x_2.$$
Он отрицателен при разных знаках и положителен при одинаковых. Поэтому правило XOR выражается линейным порогом **в новом пространстве**: $y=\mathbf1\{u<0\}$, за исключением несущественного для непрерывного распределения соглашения на осях.

Полином степени 2 даёт
$$\phi(x)=(x_1,x_2,x_1^2,x_1x_2,x_2^2).$$
Логистическая регрессия на этих признаках остаётся линейной по $\phi(x)$, но её граница в исходной плоскости может быть кривой. Например, логит $-a x_1x_2$, $a>0$, соответствует границе из двух осей.

Произведение $x_1x_2$ называют взаимодействием признаков. Добавление только квадратов без этого произведения не позволяет точно выразить XOR на всех четырёх квадрантах. В практикуме это проверяется сравнением границ решений.

### 4.2. Ядра в SVM
Другой способ работать с новыми признаками — не строить их явно. В ядровом SVM предсказание можно записать как
$$f(x)=\sum_{i=1}^{n}\alpha_i s_i K(x_i,x)+b,$$
где ненулевые $\alpha_i$ соответствуют опорным объектам, а
$$K(x,x')=\langle\phi(x),\phi(x')\rangle.$$
Ядро позволяет вычислить скалярное произведение в новом пространстве, не выписывая сам вектор $\phi(x)$. Не любая произвольная функция сходства является допустимым ядром: требуется положительная полуопределённость соответствующих матриц Грама.

Для RBF-ядра
$$K(x,x')=\exp(-\gamma\|x-x'\|^2).$$
Близкие объекты имеют значение ядра около 1, далёкие — около 0. При большом $\gamma$ влияние опорного объекта становится более локальным; граница может изменяться на более коротких расстояниях. Параметр $C$ по-прежнему отвечает за цену нарушений зазора. Это разные параметры, их подбирают совместно по validation.

Расстояния зависят от масштаба координат, поэтому стандартизация важна и здесь. Большое $\gamma$ повышает риск подгонки локальных особенностей, но не гарантирует переобучение на каждой конкретной выборке.

## 5. Как оценивать качество классификации
### 5.1. Матрица ошибок: сначала фиксируем положительный класс
Пусть класс 1 — событие, которое хотим находить. Строки матрицы — истинные метки, столбцы — предсказанные:

| | Предсказан 0 | Предсказан 1 |
|---|---:|---:|
| Истинный 0 | TN: верно отвергли | FP: ложная тревога |
| Истинный 1 | FN: пропустили событие | TP: верно нашли |

Рассмотрим 100 объектов: 20 положительных и 80 отрицательных. Модель нашла 15 положительных, пропустила 5 и ошибочно назвала положительными 10 отрицательных. Тогда
$$TP=15,\quad FN=5,\quad FP=10,\quad TN=70.$$
Одна матрица описывает **один конкретный порог**. При изменении порога её элементы меняются.

### 5.2. Accuracy, precision, recall и F1
**Accuracy** — доля всех правильных решений:
$$\operatorname{accuracy}=\frac{TP+TN}{n}=\frac{85}{100}=0.85.$$
Она понятна, но усредняет успех по всему набору. Если положительных объектов только 1%, постоянный ответ 0 даст accuracy 99%, не найдя ни одного события.

**Precision** — доля верных среди положительных решений:
$$\operatorname{precision}=\frac{TP}{TP+FP}=\frac{15}{25}=0.60.$$
Из 25 срабатываний 15 полезны. Precision важна, когда ложные тревоги расходуют ресурсы.

**Recall**, или полнота, — доля найденных среди всех положительных объектов:
$$\operatorname{recall}=\frac{TP}{TP+FN}=\frac{15}{20}=0.75.$$
Из 20 событий найдено 15. Recall важна, когда пропуск события дорог.

**F1** — гармоническое среднее precision и recall:
$$F_1=\frac{2PR}{P+R}=\frac{2TP}{2TP+FP+FN}=\frac{30}{45}\approx0.667.$$
Она требует, чтобы обе величины были достаточно большими, но не учитывает TN и не задаёт универсальную цену ошибок. Поэтому максимизация F1 не всегда соответствует прикладной цели.

При нулевых знаменателях отдельные метрики не определены; библиотечное соглашение нужно задавать явно. Например, `zero_division=0` возвращает 0 в соответствующем случае.

### 5.3. Порог и цена ошибок
При снижении порога больше объектов получают класс 1. Числа TP и FP не уменьшаются, FN не увеличивается; recall не уменьшается. Precision при этом **не обязана меняться монотонно**: важно, какие именно объекты добавились.

Если $p=P(y=1\mid x)$ известно и верные решения имеют нулевую стоимость, ожидаемая цена ответа 1 равна $c_{FP}(1-p)$, а ответа 0 — $c_{FN}p$. Выбираем класс 1, когда
$$c_{FP}(1-p)\le c_{FN}p
\iff p\ge\frac{c_{FP}}{c_{FP}+c_{FN}}.$$
При одинаковой цене ошибок получаем порог 0.5. Если пропуск в четыре раза дороже ложной тревоги, порог равен $1/(1+4)=0.2$.

Этот вывод предполагает корректные вероятности для распределения применения модели. При неточных вероятностях или иной цели порог подбирают на validation. Порог — часть окончательного решения, а не параметр, который можно многократно выбирать по test.

### 5.4. ROC-кривая и ROC AUC: качество порядка
Для каждого порога вычисляем
$$TPR=\frac{TP}{TP+FN}=\operatorname{recall},\qquad
FPR=\frac{FP}{FP+TN}.$$
ROC-кривая показывает TPR по вертикали и FPR по горизонтали при переборе порога. Идеальный порядок позволяет найти все положительные объекты до первой ложной тревоги: кривая проходит через левый верхний угол.

ROC AUC — площадь под этой кривой. Её удобно интерпретировать через случайную пару объектов:
$$\operatorname{AUC}=P(f(x^+)>f(x^-))+\tfrac12P(f(x^+)=f(x^-)).$$
Пример: scores положительных объектов равны 0.9 и 0.6, отрицательных — 0.7 и 0.2. Из четырёх положительно-отрицательных пар три упорядочены правильно, значит ROC AUC равна $3/4$.

В метрику нужно передавать **scores, а не готовые классы**: бинаризация уничтожает большую часть информации о порядке. Строго возрастающее преобразование scores сохраняет AUC.

На популяционном уровне ROC зависит от условных распределений scores внутри классов, а не от их долей. Но низкий FPR на огромном отрицательном классе всё ещё может означать много ложных тревог: высокий AUC не гарантирует полезную precision.

### 5.5. PR-кривая и average precision
PR-кривая показывает precision по вертикали и recall по горизонтали. Она отвечает на вопрос: «Какую долю положительных объектов можем найти при заданном качестве срабатываний?»

Для неселективного отбора базовый уровень precision соответствует доле положительного класса $\pi=P(y=1)$. Поэтому результаты PR-сравнений нужно интерпретировать с учётом этой доли. При редких положительных событиях PR-кривая часто лучше показывает нагрузку от ложных тревог.

**Average precision (AP)** усредняет precision с весами прироста recall:
$$AP=\sum_k(R_k-R_{k-1})P_k,$$
где пороги упорядочены так, чтобы recall возрастал. AP не обязательно равна площади под PR-кривой, посчитанной методом трапеций.

В предыдущем примере порядок меток по убыванию score: $1,0,1,0$. Первый положительный объект даёт precision 1 при приросте recall $1/2$, второй — precision $2/3$ при таком же приросте. Поэтому
$$AP=\tfrac12\cdot1+\tfrac12\cdot\tfrac23=\tfrac56.$$
Как ROC AUC, AP оценивает ранжирование и требует непрерывных scores или вероятностей, а не выбранного порога.

### 5.6. Хорошее ранжирование не равно хорошим вероятностям
Возьмём два объекта с метками $(1,0)$. Модель A выдаёт вероятности $(0.6,0.4)$, модель B — $(0.99,0.98)$. Обе правильно ставят положительный объект выше отрицательного: ROC AUC равна 1.

Но средняя log loss существенно различается:
$$J_A=-\tfrac12(\log0.6+\log0.6)\approx0.511,$$
$$J_B=-\tfrac12(\log0.99+\log0.02)\approx1.961.$$
Модель B приписала отрицательному объекту вероятность класса 1, близкую к единице. Правильный порядок не спасает её от большой вероятностной ошибки.

**Калибровка** означает соответствие заявленной вероятности фактической частоте события: среди объектов с прогнозом около 0.7 доля положительных должна быть около 70%. На практике её проверяют по группам предсказаний на отложенных данных, учитывая размер групп. Log loss оценивает вероятностный прогноз в целом и не является отдельной метрикой только калибровки.

## 6. Как провести честное сравнение
1. **Train:** обучаем веса и преобразования признаков, включая параметры стандартизации.
2. **Validation:** выбираем модель, $C$, $\gamma$, степень полинома и порог, если он нужен.
3. **Test:** после фиксации решений получаем итоговую оценку обобщения.

Если после просмотра test поменять модель и снова выбрать лучший результат по test, он начинает играть роль validation. Даже хороший алгоритм можно переоптимизировать под небольшую проверочную выборку множеством попыток.

Для воспроизводимого сравнения используем одинаковые разбиения и одну заранее выбранную целевую метрику. Остальные метрики и графики помогают понять причины результата. Небольшое преимущество на одном разбиении ещё не доказывает устойчивого превосходства.

Связь с практикумом: реализация log loss проверяет математику и устойчивость; разделимая выборка выявляет разницу между правильными метками и минимумом потери; XOR показывает влияние пространства признаков и формы границы.

## 7. Дополнительные подробности
### 7.1. Выпуклость логистической задачи
Для одного объекта
$$\frac{\partial^2\ell}{\partial z^2}=p(1-p)\ge0.$$
Пусть $A=[X\ \mathbf1]$, $\theta=(w,b)$, $D=\operatorname{diag}(p_i(1-p_i))$. Тогда
$$\nabla^2J_\lambda(\theta)=\frac1n A^\top DA+\lambda\operatorname{diag}(1,\ldots,1,0).$$
Для любого вектора $u$ квадратичная форма первого слагаемого равна $\sum_iD_{ii}(a_i^\top u)^2/n\ge0$; второе тоже неотрицательно. Следовательно, задача выпуклая.

Выпуклость исключает неглобальные локальные минимумы, но не гарантирует существование конечного минимума или его единственность. Строгая разделимость без штрафа — уже рассмотренный пример отсутствия конечного минимума.

### 7.2. Почему нельзя просто приравнять C и λ
Наша запись использует среднюю потерю:
$$\frac1n\sum_i\ell_i+\frac\lambda2\|w\|^2.$$
Запись вида $\tfrac12\|w\|^2+C\sum_i\ell_i$ после деления на $Cn$ становится
$$\frac1n\sum_i\ell_i+\frac{1}{2Cn}\|w\|^2.$$
Для этих соглашений $\lambda=1/(Cn)$. В первом упражнении используется именно такое сопоставление с библиотечной логистической регрессией без весов объектов и классов.

Соотношение нужно получать из фактической целевой функции: сумма или среднее потерь, нормировка весов наблюдений и штрафование сдвига могут различаться. Одинаковые числовые значения параметров разных API сами по себе не означают одинаковую регуляризацию.